# VeriPet Dogs - Sweep de Losses para Verificacao

Notebook novo para testar variacao de seeds e losses modernas sem abusar da base PetFace Dog completa.

Fluxo:
1. baseline congelado ConvNeXt-S + cosine similarity;
2. triagem em sample menor de treino/validacao;
3. promocao dos 2 melhores candidatos para rodada robusta com 5 seeds.

In [ ]:
from pathlib import Path
from pprint import pprint
import subprocess
import sys

DOG_REPO_URL = "https://github.com/mdrapha/veripet-dog.git"
REPO_DIR = Path("/content/veripet-dog")
LOCAL_CANDIDATES = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]

def _add_src_path(repo_dir):
    src_dir = repo_dir / "src"
    if src_dir.exists() and str(src_dir) not in sys.path:
        sys.path.insert(0, str(src_dir))

for candidate in LOCAL_CANDIDATES:
    if (candidate / "src" / "veripet").exists():
        _add_src_path(candidate)

try:
    import veripet  # noqa: F401
except ModuleNotFoundError:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", DOG_REPO_URL, str(REPO_DIR)], check=True)
    elif (REPO_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
    _add_src_path(REPO_DIR)

try:
    import optuna  # noqa: F401
except ModuleNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna"], check=True)


In [ ]:
from veripet.research.dog_verification_workflows import (
    DogVerificationPaths,
    DogVerificationSweepConfig,
    run_dog_verification_sweep,
    run_promoted_robust_sweep,
)

PATHS = DogVerificationPaths(
    drive_base=Path('/content/drive/MyDrive/dogs'),
    work_dir=Path('/content/dogs'),
    results_dir=Path('/content/drive/MyDrive/dogs/verification_experiments'),
)


## Dry Run

Confere configuracao e artefatos esperados sem carregar imagens.

In [ ]:
dry = run_dog_verification_sweep(
    DogVerificationSweepConfig(
        paths=PATHS,
        run_name='dry_run_loss_sweep',
        dry_run=True,
    )
)
pprint(dry)


## Triagem Barata

Usa 25% das identidades do treino e 25% dos pares de validacao/teste, preservando amostragem estratificada por tipo de par. Seeds de triagem: `[42, 101, 2026]`.

In [ ]:
triage_summary = run_dog_verification_sweep(
    DogVerificationSweepConfig(
        paths=PATHS,
        run_name='triage_loss_sweep_25pct',
        train_identity_fraction=0.25,
        pair_fraction=0.25,
        losses=['Softmax', 'ArcFace', 'TripletMarginLoss', 'CosFace', 'AdaFace', 'MagFace'],
        seeds=[42, 101, 2026],
        epochs=8,
        batch_size=128,
        eval_batch_size=256,
        num_workers=4,
        use_amp=True,
    )
)
pprint(triage_summary)


## Rodada Robusta Promovida

Promove automaticamente os 2 melhores losses da triagem por `val_auc`, desempate por `negative_same_breed_accuracy`. Usa 5 seeds e 20 epocas.

In [ ]:
candidate_summary_json = Path(triage_summary['results_root']) / 'verification_summary.json'
robust_summary = run_promoted_robust_sweep(
    candidate_summary_json=candidate_summary_json,
    paths=PATHS,
    seeds=[7, 13, 42, 101, 2026],
    epochs=20,
    train_identity_fraction=1.0,
    pair_fraction=1.0,
)
pprint(robust_summary)
